In [ ]:
#Kaidee Scraper urls

import time
import csv
import re
import os
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

START_URL = "https://baan.kaidee.com/c2-realestate?sort=latest"
OUTPUT_CSV_FILE = "kaidee_listing_urls.csv"
PAGE_TIMEOUT = 40
MAX_PAGES = 1000

PATTERN = re.compile(r"https://baan\.kaidee\.com/product-\d+")

def deep_scroll(driver, rounds=12, pause=0.8):
    prev = 0
    for _ in range(rounds):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(pause)
        cur = driver.execute_script("return document.body.scrollHeight")
        if cur == prev:
            break
        prev = cur

def extract_links(html):
    urls = set(PATTERN.findall(html))
    return sorted(urls)

def wait_for_page_load(driver, wait):
    print("  -> Waiting for listings to load...")
    wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'a[href*="/product-"]')))
    time.sleep(2)
    print("  -> Listings loaded")

def main():
    options = uc.ChromeOptions()
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-notifications")
    options.add_argument("--disable-blink-features=AutomationControlled")
    
    driver = uc.Chrome(options=options, version_main=145)
    wait = WebDriverWait(driver, 15)

    all_urls = set()
    current_url = START_URL
    page = 1

    while page <= MAX_PAGES:
        print(f"\n[Page {page}] Loading: {current_url}")
        driver.get(current_url)
        time.sleep(3)
        
        wait_for_page_load(driver, wait)
        deep_scroll(driver, rounds=15, pause=0.8)
        time.sleep(1)

        html = driver.page_source
        links = extract_links(html)

        if not links:
            print("  -> No listings found")
            break

        all_urls.update(links)
        print(f"  -> Found {len(links)} listings (total: {len(all_urls)})")

        try:
            print("  -> Looking for next page button...")
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)
            
            next_btn = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'button[data-testid="next"]')))
            parent_anchor = next_btn.find_element(By.XPATH, './parent::a[@href]')
            next_page_url = parent_anchor.get_attribute('href')
            
            if not next_page_url:
                print("  -> Next page URL not found")
                break
            
            print(f"  -> Found next page: {next_page_url}")
            print("  -> Navigating to next page...")
            current_url = next_page_url
            page += 1
            time.sleep(1)
            
        except Exception as e:
            print(f"  -> No more pages (error: {type(e).__name__})")
            break

    driver.quit()

    with open(OUTPUT_CSV_FILE, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["ListingURL"])
        for u in sorted(all_urls):
            w.writerow([u])
    
    print(f"\n✓ Done. {len(all_urls)} listings saved to {OUTPUT_CSV_FILE}")

if __name__ == "__main__":
    main()

In [9]:
#kaidee scraped details

import csv
import time
import re
import socket
from pathlib import Path
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import WebDriverException, TimeoutException
from urllib3.exceptions import ReadTimeoutError, ConnectionError as Urllib3ConnectionError

INPUT_CSV_FILE = "kaidee_listing_urls.csv"
OUTPUT_CSV_FILE = "kaidee_scraped_details.csv"
WAIT = 25
MAX_RETRIES = 3
NAVIGATION_TIMEOUT = 60

def is_valid_url(url):
    if not url or not isinstance(url, str):
        return False
    url = url.strip()
    if not url.startswith(('http://', 'https://')):
        return False
    if len(url) < 10:
        return False
    return True

def click_cookie(driver):
    try:
        btns = driver.find_elements(By.XPATH, "//button[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'understand') or contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'accept') or contains(text(), 'ยอมรับ')]")
        if btns:
            driver.execute_script("arguments[0].click();", btns[0])
            time.sleep(0.5)
    except:
        pass

def scrape(driver, url):
    if not is_valid_url(url):
        print(f"INVALID_URL: {url}")
        return {"Post_URL": url, "Full_Content": "INVALID_URL"}
    
    for attempt in range(MAX_RETRIES):
        try:
            driver.set_page_load_timeout(NAVIGATION_TIMEOUT)
            driver.get(url)
            break
        except (ReadTimeoutError, Urllib3ConnectionError, socket.timeout) as e:
            error_type = type(e).__name__
            print(f"TIMEOUT_ATTEMPT_{attempt + 1}/{MAX_RETRIES} for {url}: {error_type}")
            if attempt < MAX_RETRIES - 1:
                time.sleep(2 ** attempt)
            else:
                print(f"TIMEOUT_FINAL: {url}")
                return {"Post_URL": url, "Full_Content": f"TIMEOUT: {error_type}"}
        except (WebDriverException, TimeoutException) as e:
            error_type = type(e).__name__
            print(f"NAVIGATION_ERROR for {url}: {error_type}")
            return {"Post_URL": url, "Full_Content": f"NAV_ERROR: {error_type}"}
        except Exception as e:
            error_type = type(e).__name__
            print(f"UNEXPECTED_ERROR for {url}: {error_type}")
            return {"Post_URL": url, "Full_Content": f"ERROR: {error_type}"}
    
    w = WebDriverWait(driver, WAIT)
    
    click_cookie(driver)
    
    try:
        w.until(EC.presence_of_element_located((By.TAG_NAME, "h1")))
    except Exception as e:
        print(f"Error loading {url}: {e}")
        return {"Post_URL": url, "Full_Content": "N/A"}
        
    time.sleep(2)
    
    title = driver.execute_script("""
        var h1 = document.querySelector('h1');
        return h1 ? h1.innerText.trim() : '';
    """)
    
    price = driver.execute_script("""
        var p = document.querySelector("span[font-size='32px']");
        return p ? p.innerText.trim() : '';
    """)
    
    attributes = driver.execute_script("""
        var items = document.querySelectorAll("ul#has-attributes li");
        var parts = [];
        for (var i = 0; i < items.length; i++) {
            var label = items[i].querySelector("span:not([type='Body4-Kaidee-Regular'])");
            var val = items[i].querySelector("b");
            var labelTxt = "";
            if (!label) {
                var spans = items[i].querySelectorAll("span");
                if (spans.length > 0) labelTxt = spans[0].innerText.trim();
            } else {
                labelTxt = label.innerText.trim();
            }
            if (val) {
                parts.push(labelTxt + ": " + val.innerText.trim());
            }
        }
        return parts.join(' | ');
    """)
    
    update_and_location = driver.execute_script("""
        var uls = document.querySelectorAll("ul");
        for (var i = 0; i < uls.length; i++) {
            var items = uls[i].querySelectorAll("li");
            var hasUpdate = false;
            for(var j=0; j < items.length; j++){
                if(items[j].innerText.includes('อัพเดทล่าสุด')) {
                    hasUpdate = true;
                    break;
                }
            }
            if(hasUpdate) {
                var parts = [];
                for(var j=0; j < items.length; j++){
                    var spans = items[j].querySelectorAll("span");
                    if(spans.length >= 2) {
                        parts.push(spans[0].innerText.trim() + ": " + spans[1].innerText.trim());
                    }
                }
                return parts.join(' | ');
            }
        }
        return '';
    """)

    driver.execute_script("""
        var btns = document.querySelectorAll('a, button');
        for (var i = 0; i < btns.length; i++) {
            var t = btns[i].innerText.trim();
            if (t === 'อ่านเพิ่มเติม') {
                btns[i].click();
                break;
            }
        }
    """)
    time.sleep(1)
    
    description = driver.execute_script("""
        var p = document.querySelector("p.inner-text");
        return p ? p.innerText.trim() : '';
    """)
    
    phone_els = driver.find_elements(By.CSS_SELECTOR, "span.masked[data-value]")
    phones = sorted({e.get_attribute("data-value").strip() for e in phone_els if e.get_attribute("data-value")})
    phone_str = "เบอร์โทร: " + ", ".join(phones) if phones else ""
    
    parts = [p for p in [title, f"ราคา: {price}", attributes, update_and_location, description, phone_str] if p]
    full_content = "\n".join(parts)
    
    return {"Post_URL": url, "Full_Content": full_content}

def main():
    options = uc.ChromeOptions()
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-notifications")
    options.add_argument("--disable-blink-features=AutomationControlled")
    driver = uc.Chrome(options=options, version_main=145)
    driver.set_page_load_timeout(NAVIGATION_TIMEOUT)
    
    with open(INPUT_CSV_FILE, "r", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        urls = []
        for row in reader:
            if 'URL' in row:
                urls.append(row['URL'].strip())
            elif 'PostURL' in row:
                urls.append(row['PostURL'].strip())
            elif 'Post_URL' in row:
                urls.append(row['Post_URL'].strip())
            else:
                urls.append(list(row.values())[0].strip())
    
    START_INDEX = 6043
    urls = urls[START_INDEX:]
    
    out_path = Path(OUTPUT_CSV_FILE)
    file_exists = out_path.exists() and out_path.stat().st_size > 0
    
    with open(OUTPUT_CSV_FILE, "a", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=["Post_URL", "Full_Content"])
        if not file_exists:
            w.writeheader()
            f.flush()
            
        count = START_INDEX
        for u in urls:
            count += 1
            if not u: continue
            print(f"[{count}] Scraping: {u}")
            data = scrape(driver, u)
            w.writerow(data)
            f.flush()
            time.sleep(1.5)
            
    driver.quit()
    print("Done")

if __name__ == "__main__":
    main()


[6044] Scraping: https://baan.kaidee.com/product-370622249
[6045] Scraping: https://baan.kaidee.com/product-370622250
[6046] Scraping: https://baan.kaidee.com/product-370622252
[6047] Scraping: https://baan.kaidee.com/product-370622254
[6048] Scraping: https://baan.kaidee.com/product-370622255
[6049] Scraping: https://baan.kaidee.com/product-370622256
[6050] Scraping: https://baan.kaidee.com/product-370622257
[6051] Scraping: https://baan.kaidee.com/product-370622258
[6052] Scraping: https://baan.kaidee.com/product-370622259
[6053] Scraping: https://baan.kaidee.com/product-370622260
[6054] Scraping: https://baan.kaidee.com/product-370622262
[6055] Scraping: https://baan.kaidee.com/product-370622266
[6056] Scraping: https://baan.kaidee.com/product-370622270
[6057] Scraping: https://baan.kaidee.com/product-370622274
[6058] Scraping: https://baan.kaidee.com/product-370622278
[6059] Scraping: https://baan.kaidee.com/product-370622279
[6060] Scraping: https://baan.kaidee.com/product-3706222